# Preprocessing

In [1]:
import sys; sys.path.append("..")
import numpy as np
import pandas as pd
from pathlib import Path
from src.aggregate import aggregate
from src.moments import group_kurt

RAW = Path("../data/raw")
INTERIM = Path("../data/interim")

In [2]:
ins = pd.read_csv(RAW / "installments_payments.csv")
ins["days_late"] = ins["DAYS_ENTRY_PAYMENT"] - ins["DAYS_INSTALMENT"]
ins["payment_ratio"] = ins["AMT_PAYMENT"] / ins["AMT_INSTALMENT"]
ins["underpayment"] = ins["AMT_INSTALMENT"] - ins["AMT_PAYMENT"]
ins["is_late"] = (ins["days_late"] > 0).astype("int8")
ins["is_short"] = (ins["underpayment"] > 0).astype("int8")
ins = ins.replace([np.inf, -np.inf], np.nan)
ins.shape

(13605401, 13)

# Feature Engineering

## Iteration 1

In [3]:
i = aggregate(ins, "SK_ID_CURR", "ins", ignore=["SK_ID_PREV"])

In [4]:
ins = ins.sort_values(["SK_ID_PREV", "NUM_INSTALMENT_NUMBER"]).reset_index(drop=True)
ins["blk"] = (ins["SK_ID_PREV"].ne(ins["SK_ID_PREV"].shift()) | ins["is_late"].ne(ins["is_late"].shift())).cumsum()
runs = ins[ins["is_late"] == 1].groupby(["SK_ID_PREV", "blk"]).size()
loan = pd.DataFrame({"curr": ins.groupby("SK_ID_PREV")["SK_ID_CURR"].first(), "streak": runs.groupby("SK_ID_PREV").max()})
loan["streak"] = loan["streak"].fillna(0)
gc = loan.groupby("curr")
i = i.join(pd.DataFrame({"ins_n_loans": gc.size(), "ins_max_late_streak": gc["streak"].max()}))
i = i.join(ins.loc[ins["DAYS_INSTALMENT"] >= -365].groupby("SK_ID_CURR")["is_late"].mean().rename("ins_late_share_1y"))
i.shape

(339587, 59)

In [5]:
parts = []
for d in [180, 365, 730, 1500]:
    sub = ins[ins["DAYS_INSTALMENT"] >= -d]
    w = sub.groupby("SK_ID_CURR")
    parts += [w["days_late"].max().rename(f"ins_late_max_{d}"),
              w["days_late"].mean().rename(f"ins_late_mean_{d}"),
              w["days_late"].skew().rename(f"ins_late_skew_{d}"),
              group_kurt(sub["days_late"], sub["SK_ID_CURR"]).rename(f"ins_late_kurt_{d}"),
              w["is_late"].mean().rename(f"ins_late_share_{d}"),
              w["underpayment"].mean().rename(f"ins_underpay_{d}"),
              w["is_short"].mean().rename(f"ins_short_share_{d}")]
windows = pd.concat(parts, axis=1)

frac = {}
for s, l in [(180, 365), (365, 730), (730, 1500)]:
    frac[f"ins_late_share_frac_{s}_{l}"] = windows[f"ins_late_share_{s}"] / windows[f"ins_late_share_{l}"]
frac = pd.DataFrame(frac).replace([np.inf, -np.inf], np.nan)

g = ins.groupby("SK_ID_CURR")
dx = ins["DAYS_INSTALMENT"] - g["DAYS_INSTALMENT"].transform("mean")
dy = ins["days_late"] - g["days_late"].transform("mean")
trend = (dx * dy).groupby(ins["SK_ID_CURR"]).sum() / (dx * dx).groupby(ins["SK_ID_CURR"]).sum()

recent = ins.sort_values("DAYS_INSTALMENT").groupby("SK_ID_CURR").tail(1)[["SK_ID_CURR", "SK_ID_PREV"]]
ll = ins.merge(recent, on=["SK_ID_CURR", "SK_ID_PREV"]).groupby("SK_ID_CURR")
lastloan = pd.DataFrame({"ins_last_loan_late_max": ll["days_late"].max(),
                         "ins_last_loan_late_mean": ll["days_late"].mean(),
                         "ins_last_loan_late_share": ll["is_late"].mean(),
                         "ins_last_loan_pay_ratio": ll["payment_ratio"].mean()})

i = i.join(windows).join(frac).join(trend.replace([np.inf, -np.inf], np.nan).rename("ins_days_late_trend")).join(lastloan)
i.shape

(339587, 95)

## Iteration 2

In [6]:
for flag, tag in [(1, "late"), (0, "ontime")]:
    w = ins[ins["is_late"] == flag].groupby("SK_ID_CURR")
    i = i.join(pd.DataFrame({
        f"ins_{tag}_underpay_mean": w["underpayment"].mean(),
        f"ins_{tag}_payratio_mean": w["payment_ratio"].mean(),
        f"ins_{tag}_amt_mean": w["AMT_PAYMENT"].mean(),
        f"ins_{tag}_days_late_mean": w["days_late"].mean(),
    }))
i = i.replace([np.inf, -np.inf], np.nan)
i.shape

(339587, 103)

# Save

In [7]:
i.reset_index().to_pickle(INTERIM / "installments_payments.pkl")
i.shape

(339587, 103)